In [53]:
import pandas as pd
import re


# pandas 3.0 uses new Arrow string type by default which crashes 
# this setting reverts to old object dtype which is compatible
pd.options.future.infer_string = False


df = pd.read_csv('../data/raw/locations.csv')
print("Shape:", df.shape)
df.head()

Shape: (5000, 9)


,location_id,street_address,city,state,zip_code,county,latitude,longitude,metro_area
0,LOC-0001,3136 Nicholas Forges,Arlington,TX,53583,Jones County,33.102878,-104.702806,Dallas-Fort Worth Metro
1,LOC-0002,62397 Paul Dale Suite 354,Elizabeth,NJ,72722,Harris County,43.664447,-120.403269,New York Metro
2,LOC-0003,90421 Ellis Extension,aurora,IL,36144,Wolf County,32.258704,-106.325675,Chicago Metro
3,LOC-0004,9292 Christopher Walks,columbus,GA,94790,Boyd County,33.210406,-79.492031,Atlanta Metro
4,LOC-0005,774 Jessica Street,Arlington,TX,62431,Salas County,45.576662,-101.961248,Dallas-Fort Worth Metro


In [54]:
df.shape

(5000, 9)

In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   location_id     5000 non-null   object 
 1   street_address  5000 non-null   object 
 2   city            5000 non-null   object 
 3   state           5000 non-null   object 
 4   zip_code        4895 non-null   object 
 5   county          5000 non-null   object 
 6   latitude        4897 non-null   float64
 7   longitude       4885 non-null   float64
 8   metro_area      5000 non-null   object 
dtypes: float64(2), object(7)
memory usage: 351.7+ KB


In [56]:
df.isnull().sum()

location_id         0
street_address      0
city                0
state               0
zip_code          105
county              0
latitude          103
longitude         115
metro_area          0
dtype: int64

In [57]:
df.duplicated().sum()

np.int64(0)

In [58]:
# Fill missing lat/lon with city+state group centroid (mean)
city_state_mean = df.groupby(['city', 'state'])[['latitude', 'longitude']].transform('mean')
df['latitude'] = df['latitude'].fillna(city_state_mean['latitude'])
df['longitude'] = df['longitude'].fillna(city_state_mean['longitude'])

# Fill missing zip_code with "(NULL)"
df['zip_code'] = df['zip_code'].fillna('(NULL)')

print("Remaining nulls:")
print(df[['latitude', 'longitude', 'zip_code']].isnull().sum())

Remaining nulls:
latitude     2
longitude    0
zip_code     0
dtype: int64


In [59]:
# Drop the 2 rows where latitude is still null (unique city+state, no group mean available)
df = df.dropna(subset=['latitude'])

print("Shape after drop:", df.shape)
print("Remaining nulls:\n", df.isnull().sum())

Shape after drop: (4998, 9)
Remaining nulls:
 location_id       0
street_address    0
city              0
state             0
zip_code          0
county            0
latitude          0
longitude         0
metro_area        0
dtype: int64


In [60]:
# Clean text columns
title_cols = ['city', 'street_address', 'county', 'metro_area']
for col in title_cols:
    df[col] = df[col].str.strip().str.replace(r'\s+', ' ', regex=True).str.title()

df['state'] = df['state'].str.strip().str.upper()

df[['street_address', 'city', 'state', 'county', 'metro_area']].head(10)

,street_address,city,state,county,metro_area
0,3136 Nicholas Forges,Arlington,TX,Jones County,Dallas-Fort Worth Metro
1,62397 Paul Dale Suite 354,Elizabeth,NJ,Harris County,New York Metro
2,90421 Ellis Extension,Aurora,IL,Wolf County,Chicago Metro
3,9292 Christopher Walks,Columbus,GA,Boyd County,Atlanta Metro
4,774 Jessica Street,Arlington,TX,Salas County,Dallas-Fort Worth Metro
5,6939 Robin Avenue,Columbus,GA,Butler County,Atlanta Metro
6,78068 Mendoza Union,Fort Lauderdale,FL,Horton County,Miami Metro
7,7318 Heather Lodge,Erie,PA,Dodson County,Philadelphia Metro
8,4310 Caldwell Estates,Woodbridge,NJ,Warner County,New York Metro
9,7262 Sims Squares,Vancouver,WA,Schultz County,Seattle Metro


In [61]:
# Validate latitude (-90 to 90) and longitude (-180 to 180)
invalid_lat = df[df['latitude'].lt(-90) | df['latitude'].gt(90)]
invalid_lon = df[df['longitude'].lt(-180) | df['longitude'].gt(180)]

print(f"Invalid latitude (outside -90 to 90): {len(invalid_lat)}")
if not invalid_lat.empty:
    print(invalid_lat[['location_id', 'city', 'state', 'latitude', 'longitude']])

print(f"\nInvalid longitude (outside -180 to 180): {len(invalid_lon)}")
if not invalid_lon.empty:
    print(invalid_lon[['location_id', 'city', 'state', 'latitude', 'longitude']])

Invalid latitude (outside -90 to 90): 0

Invalid longitude (outside -180 to 180): 0


In [62]:
# Validate location_id: format (LOC-XXXX) and uniqueness
invalid_format = df[~df['location_id'].str.match(r'^LOC-\d{4}$')]
duplicate_ids  = df[df['location_id'].duplicated(keep=False)]

print(f"Invalid format (not LOC-XXXX): {len(invalid_format)}")
if not invalid_format.empty:
    print(invalid_format[['location_id']].to_string())

print(f"\nDuplicate location_id: {len(duplicate_ids)}")
if not duplicate_ids.empty:
    print(duplicate_ids[['location_id']].sort_values('location_id').to_string())

Invalid format (not LOC-XXXX): 0

Duplicate location_id: 0


In [63]:
# Validate zip_code: 5-digit format and leading-zero check
real_zips = df[df['zip_code'] != '(NULL)']['zip_code']

invalid_zip = df[(df['zip_code'] != '(NULL)') & (~df['zip_code'].str.match(r'^\d{5}$'))]

# Leading-zero states: NJ, NY, CT, MA, RI, VT, NH, ME, PA start with 0
leading_zero_states = ['NJ', 'NY', 'CT', 'MA', 'RI', 'VT', 'NH', 'ME', 'PA']
short_zips = df[
    (df['state'].isin(leading_zero_states)) &
    (df['zip_code'] != '(NULL)') &
    (df['zip_code'].str.len() == 4)
]

print(f"Total zip_code entries   : {len(df)}")
print(f"  → (NULL) placeholders  : {(df['zip_code'] == '(NULL)').sum()}")
print(f"  → Valid 5-digit zips   : {df['zip_code'].str.match(r'^\d{5}$').sum()}")
print(f"\nInvalid format (not 5-digit numeric): {len(invalid_zip)}")
if not invalid_zip.empty:
    print(invalid_zip[['location_id', 'city', 'state', 'zip_code']].to_string())

print(f"\nPossible leading-zero loss (4-digit zip in 0xx state): {len(short_zips)}")
if not short_zips.empty:
    print(short_zips[['location_id', 'city', 'state', 'zip_code']].to_string())

Total zip_code entries   : 4998
  → (NULL) placeholders  : 105
  → Valid 5-digit zips   : 4412

Invalid format (not 5-digit numeric): 481
     location_id              city state    zip_code
6       LOC-0007   Fort Lauderdale    FL  23447-5843
11      LOC-0012          Paterson    NJ        2486
22      LOC-0023        Scottsdale    AZ  13718-5336
36      LOC-0037            Peoria    IL        9247
76      LOC-0077        Fort Worth    TX        3496
108     LOC-0109           Raleigh    NC        8210
117     LOC-0118           Chicago    IL        2508
140     LOC-0141            Tacoma    WA        1634
144     LOC-0145      Grand Rapids    MI        6833
145     LOC-0146      Jacksonville    FL  60209-2678
153     LOC-0154             Macon    GA  86986-5295
157     LOC-0158          Columbus    OH        8980
174     LOC-0175          Columbus    GA  25717-5984
178     LOC-0179         Ann Arbor    MI        1982
190     LOC-0191             Tampa    FL        5241
199     LOC-02

In [64]:
# Fix zip_code issues
# 1. Strip ZIP+4 extension (e.g., 23447-5843 → 23447)
df['zip_code'] = df['zip_code'].str.replace(r'-\d{4}$', '', regex=True)

# 2. Pad short numeric zips with leading zeros (e.g., 2486 → 02486)
df['zip_code'] = df['zip_code'].apply(
    lambda z: z.zfill(5) if z != '(NULL)' and z.isdigit() and len(z) < 5 else z
)

# Verify
remaining_invalid = df[(df['zip_code'] != '(NULL)') & (~df['zip_code'].str.match(r'^\d{5}$'))]
print(f"Remaining invalid after fix : {len(remaining_invalid)}")
print(f"Valid 5-digit zips          : {df['zip_code'].str.match(r'^\d{5}$').sum()}")
print(f"(NULL) placeholders         : {(df['zip_code'] == '(NULL)').sum()}")
if not remaining_invalid.empty:
    print(remaining_invalid[['location_id', 'city', 'state', 'zip_code']].to_string())

Remaining invalid after fix : 0
Valid 5-digit zips          : 4893
(NULL) placeholders         : 105


In [65]:
# Validate state abbreviations against valid US states + DC
valid_states = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA',
    'HI','ID','IL','IN','IA','KS','KY','LA','ME','MD',
    'MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
    'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC',
    'SD','TN','TX','UT','VT','VA','WA','WV','WI','WY','DC'
}

invalid_state = df[~df['state'].isin(valid_states)]

print(f"Invalid state abbreviations: {len(invalid_state)}")
if not invalid_state.empty:
    print(invalid_state[['location_id', 'city', 'state']].to_string())
else:
    print("All state values are valid US state abbreviations.")

Invalid state abbreviations: 0
All state values are valid US state abbreviations.


In [66]:
# Validate county format: must end with " County"
invalid_county = df[~df['county'].str.endswith(' County')]

print(f"Invalid county format (does not end with ' County'): {len(invalid_county)}")
if not invalid_county.empty:
    print(invalid_county[['location_id', 'city', 'state', 'county']].to_string())
else:
    print("All county values end with ' County'.")

Invalid county format (does not end with ' County'): 0
All county values end with ' County'.


In [67]:
# Validate metro_area format: must end with " Metro"
invalid_metro = df[~df['metro_area'].str.endswith(' Metro')]

print(f"Invalid metro_area format (does not end with ' Metro'): {len(invalid_metro)}")
if not invalid_metro.empty:
    print(invalid_metro[['location_id', 'city', 'state', 'metro_area']].to_string())
else:
    print("All metro_area values end with ' Metro'.")

Invalid metro_area format (does not end with ' Metro'): 0
All metro_area values end with ' Metro'.


In [68]:
# Export cleaned data
df.to_csv('../cleaned/locations_cleaned.csv', index=False)

print(f"Exported: cleaned/locations_cleaned.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Exported: cleaned/locations_cleaned.csv
Shape: (4998, 9)
Columns: ['location_id', 'street_address', 'city', 'state', 'zip_code', 'county', 'latitude', 'longitude', 'metro_area']
